In [ ]:
# Langchain Embedding API

# 라이브러리 불러오기
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings( 
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False, # 토큰 ID 대신 원문 문자열 전송
)

query_result = embeddings.embed_query('저는 배가 고파요')
print(query_result)

In [2]:
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요'
]

df = pd.DataFrame(data, columns=['text'])

In [3]:
def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row: get_embedding(row.text), axis=1)
df

,text,embedding
0,주식 시장이 급등했어요,"[0.003425918985158205, 0.012388941831886768, -..."
1,시장 물가가 올랐어요,"[0.0124553507193923, 0.023535393178462982, -0...."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.01394876092672348, 0.023344269022345543, -..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[0.009744472801685333, 0.014037452638149261, -..."
4,저는 빠른 비트를 좋아해요,"[0.031136970967054367, 0.002553999423980713, 0..."
5,최근 비트코인 가격이 많이 변동했어요,"[0.024296490475535393, 0.022296005859971046, 0..."


In [4]:
# 텍스트 임베딩 벡터로 변환하는 함수 정의
def get_embedding(text):
    return embeddings.embed_query(text)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(
    lambda row: get_embedding(row.text),
    axis=1
)

# 변환된 DataFrame 출력
df

,text,embedding
0,주식 시장이 급등했어요,"[0.003425918985158205, 0.012388941831886768, -..."
1,시장 물가가 올랐어요,"[0.0124553507193923, 0.023535393178462982, -0...."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.01394876092672348, 0.023344269022345543, -..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[0.009744472801685333, 0.014037452638149261, -..."
4,저는 빠른 비트를 좋아해요,"[0.031136970967054367, 0.002553999423980713, 0..."
5,최근 비트코인 가격이 많이 변동했어요,"[0.024296490475535393, 0.022296005859971046, 0..."


In [5]:
# 코사인 유사도 계산 함수
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

# 주어진 쿼리와 가장 유사한 상위 3개의 문서를 반환하는 함수
def return_answer_candidate(df, query):
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = get_embedding(query)

    # DataFrame의 각 문서 임베딩과 쿼리 임베딩 간의 유사도 계산
    df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))

    # 유사도가 높은 순으로 정렬하고 상위 3개 문서 선택
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc

# 예시 쿼리로 유사한 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.0124553507193923, 0.023535393178462982, -0....",0.700582
5,최근 비트코인 가격이 많이 변동했어요,"[0.024296490475535393, 0.022296005859971046, 0...",0.584192
0,주식 시장이 급등했어요,"[0.003425918985158205, 0.012388941831886768, -...",0.580358


In [6]:
# 다른 모델로 테스트
embeddings = OpenAIEmbeddings( 
    model="text-embedding-nomic-embed-text-v1.5",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(
    lambda row: get_embedding(row.text),
    axis=1
)

# 예시 쿼리로 유사한 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
5,최근 비트코인 가격이 많이 변동했어요,"[-0.03933055326342583, -0.00351810734719038, -...",0.816629
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.0006923280889168382, 0.012346523813903332,...",0.785077
0,주식 시장이 급등했어요,"[-0.00934308860450983, -0.00351078063249588, -...",0.761756


In [7]:
# instruction 실습
def get_detailed_instruct(task: str, query: str) -> str:
    return f'Instruct: {task}\nQuery:{query}'        # 공식 포맷 (Query: 뒤 공백 없음)

def get_embedding_instruct(text: str, task: str):    # get_embedding의 인스트럭션 버전
    return embeddings.embed_query(get_detailed_instruct(task, text))

def return_answer_candidate_instruct(df, query, task):
    query_embedding = get_embedding_instruct(query, task)   # 쿼리만 인스트럭션
    sim = df.embedding.apply(                                # 문서는 기존 무인스트럭션 벡터 재사용
        lambda x: cos_sim(np.array(x), np.array(query_embedding))
    )
    out = df.assign(similarity=sim)                          # df 원본 비오염 (비교 실험용)
    return out.sort_values("similarity", ascending=False).head(3)

In [8]:
query = '과일 값이 비싸다'

print('=== baseline (no instruction) ===')
print(return_answer_candidate(df, query)[['text', 'similarity']].to_string(), '\n')

tasks = {
    'A_generic'  : 'Given a query, retrieve sentences with semantically similar meaning',
    'B_price'    : 'Given a statement about rising prices, retrieve sentences about price increases or cost of living',
    'C_websearch': 'Given a web search query, retrieve relevant passages that answer the query',
}

for name, task in tasks.items():
    print(f'=== {name} ===')
    print(return_answer_candidate_instruct(df, query, task)[['text', 'similarity']].to_string(), '\n')

=== baseline (no instruction) ===
                     text  similarity
5    최근 비트코인 가격이 많이 변동했어요    0.816629
3  부동산 시장이 점점 더 복잡해지고 있어요    0.785077
0            주식 시장이 급등했어요    0.761756 

=== A_generic ===
                   text  similarity
5  최근 비트코인 가격이 많이 변동했어요    0.640949
2  전통 시장에는 다양한 물품들을 팔아요    0.623146
4        저는 빠른 비트를 좋아해요    0.606008 

=== B_price ===
                   text  similarity
5  최근 비트코인 가격이 많이 변동했어요    0.592812
2  전통 시장에는 다양한 물품들을 팔아요    0.578432
4        저는 빠른 비트를 좋아해요    0.571780 

=== C_websearch ===
                   text  similarity
5  최근 비트코인 가격이 많이 변동했어요    0.656174
2  전통 시장에는 다양한 물품들을 팔아요    0.641343
0          주식 시장이 급등했어요    0.634771 



In [9]:
commerce_tasks = {
    'D_place': 'Given a query, retrieve sentences about marketplaces or places where goods are sold',
    'E_buy'  : 'Given a query about buying a product, retrieve sentences about where to buy or sell goods',
}

# TEST 1 (순수 인스트럭션 효과): 쿼리 고정 + 상거래 인스트럭션
#   → 쿼리 자체가 "비싸다(가격)"라 가격 편향과 싸우는 어려운 테스트
print('### TEST 1: query="과일 값이 비싸다" ###')
for name, task in commerce_tasks.items():
    print(f'--- {name} ---')
    print(return_answer_candidate_instruct(df, '과일 값이 비싸다', task)[['text','similarity']].to_string(), '\n')

# TEST 2 (결정적 테스트): 장소/구매 의도 쿼리 + 상거래 인스트럭션
print('### TEST 2: query="과일을 어디서 살 수 있나요" ###')
for name, task in commerce_tasks.items():
    print(f'--- {name} ---')
    print(return_answer_candidate_instruct(df, '과일을 어디서 살 수 있나요', task)[['text','similarity']].to_string(), '\n')


### TEST 1: query="과일 값이 비싸다" ###
--- D_place ---
                     text  similarity
5    최근 비트코인 가격이 많이 변동했어요    0.624287
2    전통 시장에는 다양한 물품들을 팔아요    0.614567
3  부동산 시장이 점점 더 복잡해지고 있어요    0.599438 

--- E_buy ---
                   text  similarity
5  최근 비트코인 가격이 많이 변동했어요    0.637152
2  전통 시장에는 다양한 물품들을 팔아요    0.627356
4        저는 빠른 비트를 좋아해요    0.615220 

### TEST 2: query="과일을 어디서 살 수 있나요" ###
--- D_place ---
                     text  similarity
2    전통 시장에는 다양한 물품들을 팔아요    0.602391
5    최근 비트코인 가격이 많이 변동했어요    0.592680
3  부동산 시장이 점점 더 복잡해지고 있어요    0.586837 

--- E_buy ---
                   text  similarity
2  전통 시장에는 다양한 물품들을 팔아요    0.608949
5  최근 비트코인 가격이 많이 변동했어요    0.605023
4        저는 빠른 비트를 좋아해요    0.600823 

